In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from math import sqrt

In [2]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model = 512, n_heads = 8, d_ff = 2048, dropout = 0.1):
        super().__init__()
        # d_model : model's dimension
        # n_heads :  number of heads
        # d_ff: feedforward neural network's dimension
        
        # sublayer 1: multi-head self attention
        self.attention =  nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        
        # sublayer 2: feed forward network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        
        # layer normalization and  dropout
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        
        # attention with residual connection and layer normalization
        residual = x
        x = self.norm1(x)
        x, _ = self.attention(x, x, x, attn_mask=mask) #Q,K,V = x for self attention
        x = self.dropout(x) + residual
        
        # Feedforward network with residual connection and layer normalization
        residual = x
        x = self.norm2(x)
        x = self.ffn(x)
        x = residual + self.dropout(x)
        
        # Output
        return x

In [3]:
class Transformer(nn.Module):
    def __init__(self, vocab_size=32000, d_model=512, n_heads=8, n_layers=6, d_ff=2048, max_len=2048, dropout=0.1):
        super().__init__()
        
        # Embeddings
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.scale = sqrt(d_model)
        self.drop = nn.Dropout(dropout)
        
        # Stack of 6 identical transformer blocks
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range (n_layers)])
        
        # Final output
        self.final_form = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
        self.head.weight = self.tok_emb.weight 
        
    def forward(self, token_ids):
        B, T = token_ids.size()
        
        tok =  self.tok_emb(token_ids) * self.scale
        pos = self.pos_emb(torch.arange(T, device=token_ids.device))
        x = self.drop(tok + pos)
        
        #Casual mask to ensure that attention is only applied to previous tokens in the sequence
        mask = torch.triu(torch.ones(T, T, device=token_ids.device), diagonal=1).bool()
        
        # Pass through the stack of transformer blocks
        for block in self.blocks:
            x = block(x, mask=mask)
            
        x  = self.final_form(x)
        logits = self.head(x)
        
        return logits

In [4]:
# Define model and optimizer
model = Transformer(vocab_size=32000)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [5]:
# Prepare batch
batch = torch.randint(0, 32000, (8, 256))
inputs = batch[:, :-1]  # batch of 16 sequences of length 127
targets = batch[:, 1:]  # batch of 16 sequences of length 127

In [6]:
# Training step
model.train()
logits = model(inputs)
loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
optimizer.zero_grad()
loss.backward()
optimizer.step()

print(f"Loss: {loss.item():.4f}")

Loss: 484.6583
